In [1]:
import warnings

import skrub
from sklearn.ensemble import HistGradientBoostingClassifier
from skrub import TableVectorizer

import sempipes

warnings.filterwarnings("ignore")


dataset = skrub.datasets.fetch_midwest_survey()


responses = skrub.var("responses")
responses = responses.skb.set_description(dataset.metadata["description"])

labels = skrub.var("labels")
labels = labels.skb.set_description(dataset.metadata["target"])

responses = responses.skb.mark_as_X()
labels = labels.skb.mark_as_y()

with_demographic_features = responses.sem_gen_features(
    nl_prompt="""
        Compute additional demographics-related features, use your intrinsic knowledge about the US. 
        Take into account how the identification with the country or regions of it changed over the generations.         
        Also think about how the identification differs per class and education. The midwest is generally associated 
        with "Midwestern values" — friendliness, modesty, hard work, and community-mindedness.
    """,
    name="demographic_features",
    how_many=5,
)

with_spatial_features = responses.sem_gen_features(
    nl_prompt="""
        Compute additional geo-spatial-related features, use your intrinsic knowledge about the US.
    """,
    name="spatial_features",
    how_many=5,
)

all_features = with_demographic_features.skb.concat([with_spatial_features], axis=1)

feature_encoder = TableVectorizer()
encoded_responses = all_features.skb.apply(feature_encoder)
model = encoded_responses.skb.apply(HistGradientBoostingClassifier(), y=labels)

In [2]:
model

<Apply HistGradientBoostingClassifier>

In [3]:
from sempipes.optimisers import EvolutionarySearch, MonteCarloTreeSearch, TreeSearch, optimise_colopro

tuning_data = {"responses": dataset.X.head(n=500), "labels": dataset.y.head(n=500)}

outcomes = optimise_colopro(
    model,
    num_trials=16,
    scoring="accuracy",
    search=MonteCarloTreeSearch(),  # TreeSearch(),#EvolutionarySearch(population_size=6),
    cv=2,
    additional_env_variables=tuning_data,
)

2026-02-20 09:47:11,760 - INFO - SEMPIPES> COLOPRO> Computing pipeline summary for context-aware optimisation
2026-02-20 09:47:13,115 - INFO - SEMPIPES> COLOPRO> Processing trial 0
2026-02-20 09:47:13,159 - INFO - SEMPIPES> COLOPRO> Initialising optimisation via OPRO
2026-02-20 09:47:13,172 - INFO - SEMPIPES> COLOPRO> Creating root node
2026-02-20 09:47:13,173 - INFO - SEMPIPES> COLOPRO> Evaluating pipeline via 2-fold cross-validation
2026-02-20 09:47:25,529 - INFO - SEMPIPES> COLOPRO> Pipeline evaluation took 12.36 seconds
2026-02-20 09:47:25,530 - INFO - SEMPIPES> COLOPRO> Score changed from None to 0.78
2026-02-20 09:47:25,532 - INFO - SEMPIPES> COLOPRO> Processing trial 1
2026-02-20 09:47:25,596 - INFO - SEMPIPES> MCT_SEARCH> Expanding childless node 0
2026-02-20 09:47:25,597 - INFO - SEMPIPES> MCT_SEARCH> Trying to improve node with score 0.78
2026-02-20 09:47:25,598 - INFO - SEMPIPES> COLOPRO> Trying to improve node with score 0.78
2026-02-20 09:47:25,599 - INFO - SEMPIPES> COLOP

In [4]:
best_outcome = max(outcomes, key=lambda x: x.score)
best_outcome.score

0.92

In [5]:
from IPython.display import Code, display

display(Code(best_outcome.states.get("demographic_features")["generated_code"]))

import pandas as pd
import numpy as np

def _sem_gen_features(df: pd.DataFrame) -> pd.DataFrame:
    # --- Features from previous iteration (copied for continuity and dependency) ---

    # Feature 1: zip_first_digit
    # Extracts the first digit of the ZIP code, which broadly indicates a geographic region of the US.
    # This provides a coarse-grained location for the respondent, highly relevant for classifying their Census_Region.
    # Using pd.to_numeric with errors='coerce' to handle any non-numeric first characters gracefully,
    # then filling NaNs with a sentinel value (e.g., -1) and converting to int.
    # Input samples: 'In_what_ZIP_code_is_your_home_located': ['74070', '44106', '48185']
    df['zip_first_digit'] = pd.to_numeric(
        df['In_what_ZIP_code_is_your_home_located'].astype(str).str[0],
        errors='coerce'
    ).fillna(-1).astype(int)

    # Feature 2: is_zip_midwest_region
    # A binary flag indicating if the ZIP code's first digit falls into one of the generally accepted Midwest ranges (4, 5, 6).
    # This feature directly leverages the geo-spatial information from the ZIP code to indicate whether the respondent's
    # location is geographically within the Midwest, which is a strong predictor for self-identification as a Midwesterner
    # and their actual Census Region.
    # Input samples: 'zip_first_digit': [7, 4, 4]
    df['is_zip_midwest_region'] = df['zip_first_digit'].isin([4, 5, 6])

    # Feature 3: cleaned_self_reported_region
    # Standardizes the self-reported region column by cleaning up variations and converting to lowercase.
    # This feature standardizes the self-reported region, making it more consistent and usable. It directly reflects
    # how the respondent perceives their region, which is a key piece of information for the classification task.
    # Input samples: 'What_would_you_call_the_part_of_the_country_you_live_in_now': ['Southern', 'Midwest', 'Mid-west']
    df['cleaned_self_reported_region'] = df['What_would_you_call_the_part_of_the_country_you_live_in_now'].astype(str).str.lower()
    df['cleaned_self_reported_region'] = df['cleaned_self_reported_region'].replace({
        'mid-west': 'midwest',
        'mid west': 'midwest',
        'midewest': 'midwest',
        'the upper midwest': 'midwest',
        'east coast': 'east',
        'south central': 'south'
    })

    # Convert 'Yes'/'No' columns to numerical (1/0) for aggregation
    state_cols_to_convert = [col for col in df.columns if col.startswith('Do_you_consider_') and col.endswith('_state_as_part_of_the_Midwest')]
    for col in state_cols_to_convert:
        df[col + '_numeric'] = df[col].map({'Yes': 1, 'No': 0}).fillna(0) # Fillna(0) for safety, though description says no NaNs

    # List of states generally considered part of the Midwest
    midwest_states_numeric = [
        'Do_you_consider_Illinois_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_Indiana_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_Iowa_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_Kansas_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_Michigan_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_Minnesota_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_Missouri_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_Nebraska_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_North_Dakota_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_Ohio_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_South_Dakota_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_Wisconsin_state_as_part_of_the_Midwest_numeric'
    ]

    # List of states generally NOT considered part of the Midwest, but sometimes debated
    non_midwest_states_numeric = [
        'Do_you_consider_Arkansas_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_Colorado_state_as_part_of_the_Midwest_numeric',

In [6]:
display(Code(best_outcome.states.get("spatial_features")["generated_code"]))

import pandas as pd
import numpy as np

def _sem_gen_features(df: pd.DataFrame) -> pd.DataFrame:
    # --- Features from previous iteration (copied for continuity and dependency) ---

    # Feature 1: zip_first_digit
    # Extracts the first digit of the ZIP code, which broadly indicates a geographic region of the US.
    # This provides a coarse-grained location for the respondent, highly relevant for classifying their Census_Region.
    # Using pd.to_numeric with errors='coerce' to handle any non-numeric first characters gracefully,
    # then filling NaNs with a sentinel value (e.g., -1) and converting to int.
    # Input samples: 'In_what_ZIP_code_is_your_home_located': ['74070', '44106', '48185']
    df['zip_first_digit'] = pd.to_numeric(
        df['In_what_ZIP_code_is_your_home_located'].astype(str).str[0],
        errors='coerce'
    ).fillna(-1).astype(int)

    # Feature 2: is_zip_midwest_region
    # A binary flag indicating if the ZIP code's first digit falls into one of the generally accepted Midwest ranges (4, 5, 6).
    # This feature directly leverages the geo-spatial information from the ZIP code to indicate whether the respondent's
    # location is geographically within the Midwest, which is a strong predictor for self-identification as a Midwesterner
    # and their actual Census Region.
    # Input samples: 'zip_first_digit': [7, 4, 4]
    df['is_zip_midwest_region'] = df['zip_first_digit'].isin([4, 5, 6])

    # Feature 3: cleaned_self_reported_region
    # Standardizes the self-reported region column by cleaning up variations and converting to lowercase.
    # This feature standardizes the self-reported region, making it more consistent and usable. It directly reflects
    # how the respondent perceives their region, which is a key piece of information for the classification task.
    # Input samples: 'What_would_you_call_the_part_of_the_country_you_live_in_now': ['Southern', 'Midwest', 'Mid-west']
    df['cleaned_self_reported_region'] = df['What_would_you_call_the_part_of_the_country_you_live_in_now'].astype(str).str.lower()
    df['cleaned_self_reported_region'] = df['cleaned_self_reported_region'].replace({
        'mid-west': 'midwest',
        'mid west': 'midwest',
        'midewest': 'midwest',
        'the upper midwest': 'midwest',
        'east coast': 'east',
        'south central': 'south'
    })

    # Convert 'Yes'/'No' columns to numerical (1/0) for aggregation
    state_cols_to_convert = [col for col in df.columns if col.startswith('Do_you_consider_') and col.endswith('_state_as_part_of_the_Midwest')]
    for col in state_cols_to_convert:
        df[col + '_numeric'] = df[col].map({'Yes': 1, 'No': 0}).fillna(0) # Fillna(0) for safety, though description says no NaNs

    # List of states generally considered part of the Midwest
    midwest_states_numeric = [
        'Do_you_consider_Illinois_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_Indiana_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_Iowa_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_Kansas_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_Michigan_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_Minnesota_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_Missouri_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_Nebraska_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_North_Dakota_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_Ohio_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_South_Dakota_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_Wisconsin_state_as_part_of_the_Midwest_numeric'
    ]

    # List of states generally NOT considered part of the Midwest, but sometimes debated
    non_midwest_states_numeric = [
        'Do_you_consider_Arkansas_state_as_part_of_the_Midwest_numeric',
        'Do_you_consider_Colorado_state_as_part_of_the_Midwest_numeric',